# מפגש 10 — הרחבה מקצועית: Sampling בתוך Cross Validation
**רשות בלבד.** ב־Colab אפשר להתקין imbalanced-learn. השוו ללא איזון ל־RandomOverSampler, RandomUnderSampler ו־SMOTE.

חשוב: כל שיטת איזון מופעלת בתוך imblearn.Pipeline בזמן fit של כל קיפול, ולא על Test.

In [ ]:
%pip -q install imbalanced-learn

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report

url="https://raw.githubusercontent.com/arielb30/datasets/main/healthcare-dataset-stroke-data.csv"
df=pd.read_csv(url)
print(df.shape)
display(df.head())
print(df['stroke'].value_counts())

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
samplers={"No sampling":None,"Over":RandomOverSampler(random_state=42),"Under":RandomUnderSampler(random_state=42),"SMOTE":SMOTE(random_state=42)}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
features=["age","avg_glucose_level","bmi"]
X=df[features]
y=df["stroke"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
for name,sampler in samplers.items():
    steps=[("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]
    if sampler is not None:steps.append(("sample",sampler))
    steps.append(("knn",KNeighborsClassifier(n_neighbors=9)))
    pipe=Pipeline(steps)
    scores=cross_val_score(pipe,X_train,y_train,scoring="f1",cv=cv)
    print(name,round(scores.mean(),3))

שאלות: מה השתנה ב־F1? מה קרה ל־Recall? האם הגדלת Recall מחייבת פגיעה ב־Precision? מדוע אסור לבצע SMOTE לפני חלוקת Train/Test?